# Module 3 Zepto policy support assistant
This notebook exercises the real local MiniLM and Chroma retrieval, LangGraph routing and FastAPI endpoint in default mock mode. Source code is in `ingest.py`, `graph.py`, `main.py`, `schemas.py` and `prompts.py`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Markdown, display
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "SUBMISSION_CHECKLIST.md").exists())
print("Python:", sys.executable)
print("Project:", ROOT)


## 1 Inspect eight assignment policies and local retrieval
The two small Python programs run in separate processes so the notebook kernel stays light. `ingest.py` builds the genuine local vector index, and `demo.py` tests LangGraph and the API.

In [ ]:
import os
import subprocess
env = os.environ.copy()
env["MOCK_LLM"] = "1"
module = ROOT / "support_assistant"
for script in ["ingest.py", "demo.py"]:
    completed = subprocess.run([sys.executable, str(module / script)],
                               cwd=module, env=env, text=True,
                               capture_output=True, check=True)
    print(script, completed.stdout)


## 2 Read the structured prompt
It has role, context, task, format and length components, a negative constraint and a few-shot example. It is used only by the optional real-LLM route.

In [ ]:
sys.path.insert(0, str(module))
from prompts import POLICY_PROMPT
print(POLICY_PROMPT)


## 3 Test the graph and validated API responses
The default mode uses no LLM provider. The policy route still performs real embedding and Chroma retrieval.

In [ ]:
import json
summary = json.loads((module / "outputs" / "demo_results.json").read_text(encoding="utf-8"))
print("Indexed chunks:", summary["indexed_chunks"])
print("Top three:", summary["delivery_top_three"])
assert summary["indexed_chunks"] == 8
assert summary["delivery_top_three"][0] == "doc_01"
for name, example in summary["examples"].items():
    print(name, example["intent"], json.dumps(example["response"]))


## 4 Local server and Docker
From `support_assistant/`, run `python -m uvicorn main:app --host 127.0.0.1 --port 7860`. The module README shows the actual saved POST responses and Docker commands. On the first run, the public MiniLM weights must download and cache.